In [ ]:
!pip install -q vllm
!pip install -q "litellm>=1.0.0" datasets python-dotenv "staticfg>=0.9.5" tenacity tabulate

!pip uninstall -y -q torchaudio torchcodec

print("\ninstall done. click Runtime > Restart session, then start at cell 2.")
print("do not re-run this cell after the restart.")

In [ ]:
import os, subprocess, sys

REPO = "/content/self-consistency-in-LLMs"
GIT_URL = "https://github.com/zyanai-pub/self-consistency-in-LLMs"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", GIT_URL, REPO], check=True)

# run_evaluation reads this at import time to pick MODELS / the API bases
os.environ["EXECUTION_MODE"] = "local"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

check = "import vllm.entrypoints.openai.api_server; print('vllm import ok')"
r = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-1500:])
assert r.returncode == 0, "vllm won't import, fix that before going further"
assert os.path.isfile(os.path.join(REPO, "src", "run_evaluation.py"))

In [ ]:
# benchmark: same size/seed as run_evaluation.py so the sample set matches
SUBSET_SIZE = 200
SUBSET_SEED = 7

# cheapest first, so a session that dies early still leaves whole strategies finished
STRATEGY_ORDER = ["baseline", "esc", "seer", "ralu"]

SYSTEM1_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM1_GPU_UTIL = 0.16     # ~2.4 GiB, enough for 0.5B fp16 + a small KV cache

# system-2 candidates, restarted one at a time with whatever sys-1 left behind
MODEL_ZOO = {
    "qwen2.5-1.5b": dict(hf="Qwen/Qwen2.5-1.5B-Instruct", gpu_util=0.55),
    "qwen2.5-3b":   dict(hf="Qwen/Qwen2.5-3B-Instruct",   gpu_util=0.70),
}
MODELS = {k: v["hf"] for k, v in MODEL_ZOO.items()}

SYS2_PORT, SYS1_PORT = 8000, 8001
SYS2_API_BASE = f"http://localhost:{SYS2_PORT}/v1"
SYS1_API_BASE = f"http://localhost:{SYS1_PORT}/v1"
MAX_MODEL_LEN = 4096
DTYPE = "half"              # T4 is sm_75, no bfloat16

CONCURRENCY = 6             # samples in flight; seer/ralu fan out further inside
MAX_RUNTIME_MIN = 300       # quit cleanly before Colab takes the VM back

import os
RESULTS_DIR = os.path.join(REPO, "results", "colab")
os.makedirs(RESULTS_DIR, exist_ok=True)

assert SYSTEM1_GPU_UTIL + max(v["gpu_util"] for v in MODEL_ZOO.values()) < 0.95

print("system-2:", MODELS)
print("system-1:", SYSTEM1_MODEL)
print("samples :", SUBSET_SIZE, "seed", SUBSET_SEED)
print("results :", RESULTS_DIR)

In [ ]:
import atexit, signal, subprocess, sys, time, urllib.request

_SERVERS = {}
OPTIONAL_FLAGS = ["--disable-log-requests"]   # this flag has been renamed before


def _log_path(port):
    return f"/content/vllm_{port}.log"


def _tail(port, n=40):
    try:
        with open(_log_path(port)) as f:
            return "".join(f.readlines()[-n:])
    except FileNotFoundError:
        return "(no log)"


def _healthy(port):
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False


def stop_vllm(port):
    p = _SERVERS.pop(port, None)
    if p is None:
        return
    p.send_signal(signal.SIGINT)
    try:
        p.wait(timeout=60)
    except subprocess.TimeoutExpired:
        p.kill()
        p.wait(timeout=30)
    time.sleep(8)   # let the driver hand the VRAM back before the next profile pass
    print(f"[vllm:{port}] stopped")


def stop_all():
    for port in list(_SERVERS):
        stop_vllm(port)


atexit.register(stop_all)


def start_vllm(model, port, gpu_util, timeout=1500, with_optional=True):
    if port in _SERVERS:
        stop_vllm(port)

    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model,
        "--served-model-name", model,
        "--port", str(port),
        "--gpu-memory-utilization", str(gpu_util),
        "--dtype", DTYPE,
        "--max-model-len", str(MAX_MODEL_LEN),
        "--enforce-eager",       # no CUDA graph capture: less VRAM, quicker startup
    ]
    if with_optional:
        cmd += OPTIONAL_FLAGS

    log = open(_log_path(port), "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    _SERVERS[port] = proc
    print(f"[vllm:{port}] loading {model} (gpu_util={gpu_util}) ...")

    t0 = time.time()
    while time.time() - t0 < timeout:
        if _healthy(port):
            print(f"[vllm:{port}] ready in {time.time() - t0:.0f}s")
            return proc
        if proc.poll() is not None:
            tail = _tail(port)
            _SERVERS.pop(port, None)
            if with_optional and "unrecognized arguments" in tail:
                print(f"[vllm:{port}] retrying without the optional flags")
                return start_vllm(model, port, gpu_util, timeout, with_optional=False)
            raise RuntimeError(f"vllm on :{port} exited.\n--- log tail ---\n{tail}")
        time.sleep(5)

    stop_vllm(port)
    raise TimeoutError(f"vllm on :{port} not ready in {timeout}s.\n{_tail(port)}")


def gpu_mem():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total",
         "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
    used, total = (int(x) for x in out.split(","))
    return f"{used}/{total} MiB ({100 * used / total:.0f}%)"


print("helpers ready")

In [ ]:
# small model first: vllm sizes its KV cache off currently-free memory, so the big
# server should be the one profiling around an existing tenant, not the other way round
start_vllm(SYSTEM1_MODEL, SYS1_PORT, SYSTEM1_GPU_UTIL)
print("GPU:", gpu_mem())

In [ ]:
import contextvars, functools, json, os, threading, time
from concurrent.futures import ThreadPoolExecutor
import litellm

litellm.telemetry = False
litellm.drop_params = True     # a local vllm build rejects some of the params litellm sends

CURRENT_SAMPLE = contextvars.ContextVar("current_sample", default=None)
CURRENT_TAG = contextvars.ContextVar("current_tag", default=("?", "?"))

# SeerSC and RaLUSC spin up their own ThreadPoolExecutors, which don't carry contextvars
# across. Without this most of their tokens land with no sample attached.
if not getattr(ThreadPoolExecutor.submit, "_ctx_patched", False):
    _orig_submit = ThreadPoolExecutor.submit

    @functools.wraps(_orig_submit)
    def _ctx_submit(self, fn, /, *args, **kwargs):
        ctx = contextvars.copy_context()
        return _orig_submit(self, ctx.run, fn, *args, **kwargs)

    _ctx_submit._ctx_patched = True
    ThreadPoolExecutor.submit = _ctx_submit


def _ensure_trailing_newline(path):
    # a VM killed mid-write leaves a partial last line; appending onto it would eat a
    # second record too
    if os.path.exists(path) and os.path.getsize(path):
        with open(path, "rb+") as f:
            f.seek(-1, os.SEEK_END)
            if f.read(1) != b"\n":
                f.write(b"\n")


class RequestLog:
    # one JSONL row per litellm call, thread safe
    def __init__(self, path):
        self.path = path
        _ensure_trailing_newline(path)
        self._lock = threading.Lock()
        self._f = open(path, "a", buffering=1)

    def write(self, row):
        line = json.dumps(row)
        with self._lock:
            self._f.write(line + "\n")

    def close(self):
        with self._lock:
            self._f.close()


REQ_LOG = None      # the runner swaps this per (model, strategy)


# everything goes through ModelManager.generate_inference -> litellm.completion, so one
# wrapper here catches every request from every strategy
if not getattr(litellm.completion, "_metered", False):
    _orig_completion = litellm.completion

    @functools.wraps(_orig_completion)
    def _metered_completion(*args, **kwargs):
        model = str(kwargs.get("model", ""))
        role = "sys1" if SYSTEM1_MODEL.split("/")[-1] in model else "sys2"
        model_label, strategy = CURRENT_TAG.get()
        t0 = time.perf_counter()
        row = {
            "model_label": model_label, "strategy": strategy, "role": role,
            "served_model": model, "sample": CURRENT_SAMPLE.get(),
            "n": int(kwargs.get("n", 1) or 1),
            "prompt_tokens": 0, "completion_tokens": 0, "error": None,
        }
        try:
            resp = _orig_completion(*args, **kwargs)
        except Exception as e:
            row["error"] = f"{type(e).__name__}: {e}"[:300]
            raise
        else:
            usage = getattr(resp, "usage", None)
            row["prompt_tokens"] = int(getattr(usage, "prompt_tokens", 0) or 0)
            row["completion_tokens"] = int(getattr(usage, "completion_tokens", 0) or 0)
            row["choices"] = len(getattr(resp, "choices", []) or [])
            return resp
        finally:
            row["latency_s"] = round(time.perf_counter() - t0, 4)
            log = REQ_LOG
            if log is not None:
                log.write(row)

    _metered_completion._metered = True
    litellm.completion = _metered_completion

print("metering on")

In [ ]:
import src.run_evaluation as RE
from src.input_layer.benchmark_loader import BenchmarkLoader
from src.evaluation_module.extractor import AnswerExtractor

RE.SYSTEM1_MODEL = SYSTEM1_MODEL
RE.SYS1_API_BASE = SYS1_API_BASE
RE.SYS2_API_BASE = SYS2_API_BASE

STRATEGY_KWARGS = RE.STRATEGY_KWARGS
for k in STRATEGY_ORDER:
    print(f"  {k:9s} {STRATEGY_KWARGS[k]}")

SUBSET = BenchmarkLoader().get_random_subset(SUBSET_SIZE, SUBSET_SEED)
print(f"\n{len(SUBSET)} samples, first question:\n  {SUBSET[0]['question'][:120]}...")

In [ ]:
import asyncio, json, math, os, time

api_keys = {}   # local servers, nothing to authenticate against


def _paths(model_label, strategy, base_dir):
    stem = os.path.join(base_dir, f"{model_label}__{strategy}")
    return f"{stem}.jsonl", f"{stem}__requests.jsonl", f"{stem}__meta.json"


def _load_done(res_path):
    done = {}
    if not os.path.exists(res_path):
        return done
    with open(res_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue        # truncated tail from a killed session
            if r.get("error"):
                continue        # failed samples get another go next pass
            done[r["i"]] = r
    return done


async def run_strategy(controller, model_label, strategy, subset, base_dir,
                       concurrency, deadline_ts):
    global REQ_LOG
    res_path, req_path, meta_path = _paths(model_label, strategy, base_dir)
    done = _load_done(res_path)
    todo = [(i, item) for i, item in enumerate(subset) if i not in done]

    print(f"  {strategy:9s} {len(done)}/{len(subset)} done, {len(todo)} to go")
    if not todo:
        return done, False

    REQ_LOG = RequestLog(req_path)
    kwargs = dict(STRATEGY_KWARGS[strategy])
    sem = asyncio.Semaphore(concurrency)
    write_lock = asyncio.Lock()
    _ensure_trailing_newline(res_path)
    out_f = open(res_path, "a", buffering=1)
    stopped_early = False
    t_start = time.time()

    async def one(i, item):
        nonlocal stopped_early
        async with sem:
            if time.time() > deadline_ts:
                stopped_early = True
                return None
            CURRENT_SAMPLE.set(i)
            CURRENT_TAG.set((model_label, strategy))
            t0 = time.time()
            try:
                out = await asyncio.to_thread(
                    controller.execute_task, item["question"], strategy, **kwargs)
                pred = out.get("answer")
                row = {
                    "i": i,
                    "prediction": pred,
                    "expected": item["answer"],
                    "correct": AnswerExtractor.answers_are_equal(item["answer"], pred),
                    "paths_sampled": out.get("paths_sampled"),
                    "time_seconds": out.get("time_seconds"),
                    "entropy": out.get("entropy", out.get("system1_entropy")),
                    "error": None,
                }
            except Exception as e:
                row = {
                    "i": i, "prediction": None, "expected": item["answer"],
                    "correct": False, "paths_sampled": 0,
                    "time_seconds": round(time.time() - t0, 3),
                    "entropy": None, "error": f"{type(e).__name__}: {e}"[:300],
                }
            async with write_lock:
                out_f.write(json.dumps(row) + "\n")
                done[i] = row
                n = len(done)
                if n % 10 == 0 or n == len(subset):
                    acc = sum(bool(r["correct"]) for r in done.values()) / n
                    el = (time.time() - t_start) / 60
                    print(f"    [{n}/{len(subset)}] acc={acc:.3f} elapsed={el:.1f}m",
                          flush=True)
            return row

    try:
        await asyncio.gather(*(one(i, it) for i, it in todo))
    finally:
        out_f.close()
        REQ_LOG.close()
        REQ_LOG = None

    wall = time.time() - t_start
    meta = {"wall_seconds": 0.0, "sessions": 0, "concurrency": concurrency}
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta.update(json.load(f))
    meta["wall_seconds"] = round(meta["wall_seconds"] + wall, 2)
    meta["sessions"] += 1
    meta["concurrency"] = concurrency
    meta["kwargs"] = kwargs
    meta["completed"] = len(done)
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    if stopped_early:
        print(f"    out of time, re-run the sweep cell to pick {strategy} back up")
    return done, stopped_early


async def sweep(models, strategies, subset, base_dir, concurrency, deadline_ts):
    os.makedirs(base_dir, exist_ok=True)
    for model_label, hf_name in models.items():
        remaining = [s for s in strategies
                     if len(_load_done(_paths(model_label, s, base_dir)[0])) < len(subset)]
        if not remaining:
            print(f"\n=== {model_label}: done already, skipping ===")
            continue

        print(f"\n=== {model_label} ({hf_name}) -> {remaining} ===")

        if not _healthy(SYS1_PORT):
            print("system-1 is not answering, restarting it")
            start_vllm(SYSTEM1_MODEL, SYS1_PORT, SYSTEM1_GPU_UTIL)

        start_vllm(hf_name, SYS2_PORT, MODEL_ZOO[model_label]["gpu_util"])
        print("  GPU:", gpu_mem())
        controller = RE.build_controller(
            hf_name, api_keys, sys2_base=SYS2_API_BASE, sys1_base=SYS1_API_BASE)
        try:
            for strategy in strategies:
                _, early = await run_strategy(controller, model_label, strategy, subset,
                                              base_dir, concurrency, deadline_ts)
                if early:
                    print("deadline hit, stopping")
                    return
        finally:
            stop_vllm(SYS2_PORT)


print("runner ready")

Re-run the cell below after a disconnect (cells 2 through 7 first) and it resumes from
the checkpoints. Keep the tab open, Colab drops idle sessions after about 90 minutes.

In [ ]:
import time

DEADLINE = time.time() + MAX_RUNTIME_MIN * 60
await sweep(MODELS, STRATEGY_ORDER, SUBSET, RESULTS_DIR, CONCURRENCY, DEADLINE)
print("\npass finished")

In [ ]:
import glob, json, math, os

import pandas as pd


def _read_jsonl(path):
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


def _quantile(vals, frac):
    if not vals:
        return None
    vals = sorted(vals)
    k = min(len(vals) - 1, max(0, int(math.ceil(frac * len(vals)) - 1)))
    return round(vals[k], 3)


def aggregate(base_dir=None):
    base_dir = base_dir or RESULTS_DIR
    out = []

    for res_path in sorted(glob.glob(os.path.join(base_dir, "*__*.jsonl"))):
        stem = os.path.basename(res_path)[: -len(".jsonl")]
        if stem.endswith("__requests"):
            continue
        model_label, strategy = stem.split("__", 1)

        results = _read_jsonl(res_path)
        if not results:
            continue
        results = list({r["i"]: r for r in results}.values())   # a resume can redo an in-flight sample
        n = len(results)
        correct = sum(bool(r.get("correct")) for r in results)
        paths = [r.get("paths_sampled") or 0 for r in results]
        wall_each = [r.get("time_seconds") or 0.0 for r in results]

        # requests fired for a sample that died before its result row was written still
        # show up here, so tokens can over-count by up to CONCURRENCY samples per crash
        reqs = _read_jsonl(_paths(model_label, strategy, base_dir)[1])
        pt = sum(r.get("prompt_tokens", 0) for r in reqs)
        ct = sum(r.get("completion_tokens", 0) for r in reqs)
        sys1 = sum(r.get("prompt_tokens", 0) + r.get("completion_tokens", 0)
                   for r in reqs if r.get("role") == "sys1")
        sys2 = sum(r.get("prompt_tokens", 0) + r.get("completion_tokens", 0)
                   for r in reqs if r.get("role") == "sys2")
        req_lat = [r["latency_s"] for r in reqs if r.get("latency_s") is not None]

        meta_path = _paths(model_label, strategy, base_dir)[2]
        meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
        wall_s = meta.get("wall_seconds", sum(wall_each))

        out.append({
            "model": model_label,
            "strategy": strategy,
            "n": n,
            "accuracy": round(correct / n, 4),
            "correct": correct,
            "errors": sum(1 for r in results if r.get("error")),
            "no_answer": sum(1 for r in results if not r.get("prediction")),
            "mean_paths_sampled": round(sum(paths) / n, 2),
            "requests_per_sample": round(len(reqs) / n, 2),
            "completion_tokens_per_sample": round(ct / n, 1),
            "prompt_tokens_per_sample": round(pt / n, 1),
            "total_tokens_per_sample": round((pt + ct) / n, 1),
            "total_tokens_per_correct": round((pt + ct) / correct, 1) if correct else None,
            "sys1_tokens": sys1,
            "sys2_tokens": sys2,
            "total_tokens": pt + ct,
            "sample_wall_s_mean": round(sum(wall_each) / n, 3),
            "req_latency_s_mean": round(sum(req_lat) / len(req_lat), 3) if req_lat else None,
            "req_latency_s_p95": _quantile(req_lat, 0.95),
            "throughput_samples_per_min": round(n / (wall_s / 60), 2) if wall_s else None,
            "strategy_wall_min": round(wall_s / 60, 2),
            "request_errors": sum(1 for r in reqs if r.get("error")),
        })

    df = pd.DataFrame(out)
    if df.empty:
        return df
    order = {s: i for i, s in enumerate(STRATEGY_ORDER)}
    return (df.assign(_o=df["strategy"].map(order))
              .sort_values(["model", "_o"]).drop(columns="_o").reset_index(drop=True))


DF = aggregate()
if DF.empty:
    print("nothing yet, run the sweep first")
else:
    cols = ["model", "strategy", "n", "accuracy", "mean_paths_sampled",
            "completion_tokens_per_sample", "total_tokens_per_sample",
            "total_tokens_per_correct", "req_latency_s_mean",
            "throughput_samples_per_min", "errors"]
    print(DF[cols].to_string(index=False))
DF

In [ ]:
import os

import matplotlib.pyplot as plt

if DF.empty:
    print("nothing to plot")
else:
    markers = ["o", "s", "^", "D"]

    fig, ax = plt.subplots(figsize=(7, 5))
    for mk, (model, g) in zip(markers, DF.groupby("model")):
        ax.scatter(g["completion_tokens_per_sample"], g["accuracy"], s=90, marker=mk,
                   label=model)
        for _, r in g.iterrows():
            ax.annotate(r["strategy"], (r["completion_tokens_per_sample"], r["accuracy"]),
                        textcoords="offset points", xytext=(6, 5), fontsize=9)
    ax.set_xlabel("completion tokens per sample")
    ax.set_ylabel("accuracy")
    ax.set_title("accuracy vs generation cost")
    ax.grid(alpha=.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "accuracy_vs_cost.png"), dpi=150)
    plt.show()

    piv = DF.pivot(index="strategy", columns="model",
                   values="total_tokens_per_correct").reindex(STRATEGY_ORDER)
    ax = piv.plot(kind="bar", figsize=(7, 4.2), rot=0)
    ax.set_title("tokens per correct answer, lower is better")
    ax.set_xlabel("")
    ax.set_ylabel("total_tokens_per_correct")
    ax.grid(alpha=.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "tokens_per_correct.png"), dpi=150)
    plt.show()

In [ ]:
import os

if DF.empty:
    print("nothing to write")
else:
    DF.to_csv(os.path.join(RESULTS_DIR, "summary_metrics.csv"), index=False)
    DF.to_json(os.path.join(RESULTS_DIR, "summary_metrics.json"), orient="records", indent=2)
    print(RESULTS_DIR)
    for p in sorted(os.listdir(RESULTS_DIR)):
        print("  ", p)

In [ ]:
stop_all()
print("GPU:", gpu_mem())